In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# INITIALIZING SPARK SESSION #

spark = SparkSession.builder.appName("FAERS_PHARMA_ETL_PIPELINE").getOrCreate()

#---------------------------------------------------------------------------------------------------#

# PHASE 5 — GOLD LAYER BUSINESS TRANSFORMATION #

# SCENARIO_01: SCORECARD #

# Business Question: "Which drugs have the highest adverse event rates and what are the
#                      most dangerous reactions associated with them?"
# Used by: Drug safety officers, Regulatory affairs teams
 
SILVER_DRUG_PATH = "/Volumes/workspace/silver/faers_parquet/drug"
SILVER_REACTION_PATH = "/Volumes/workspace/silver/faers_parquet/reaction"
SILVER_OUTCOME_PATH = "/Volumes/workspace/silver/faers_parquet/outcome"
 
GOLD_OUTPUT_PATH = "/Volumes/workspace/gold/analytics/drug_safety_scorecard"
 
#------------------------------------------------------------------------------------------------------#

# READ FROM SILVER #

 
df_drug = spark.read.parquet(SILVER_DRUG_PATH).select(
    "primary_id", "case_id", "drug_name"
)
 
df_reaction = spark.read.parquet(SILVER_REACTION_PATH).select(
    "primary_id", "case_id", "reaction"
)
 
df_outcome = spark.read.parquet(SILVER_OUTCOME_PATH).select(
    "primary_id", "case_id", "outcome"
)
 
#------------------------------------------------------------------------------------------------------#

# GOLD TRANSFORMATION #
 
df_drug_reports = df_drug.dropDuplicates(["primary_id", "drug_name"])
 
# --- total_adverse_events: how many distinct reports mention this drug -----------------
df_total_events = (
    df_drug_reports
    .groupBy("drug_name")
    .agg(F.count("*").alias("total_adverse_events"))
)
 
# --- reaction-level metrics: unique_reaction_types + most_common_reaction --------------
# Join drug reports to reactions (fan-out is expected here: one report can have several
# distinct reactions). Dedup report+drug+reaction combos first so a repeated reaction
# entry within the same report isn't counted twice.
df_drug_reactions = (
    df_drug_reports
    .join(df_reaction, on=["primary_id", "case_id"], how="inner")
    .dropDuplicates(["primary_id", "drug_name", "reaction"])
)
 
df_unique_reactions = (
    df_drug_reactions
    .groupBy("drug_name")
    .agg(F.countDistinct("reaction").alias("unique_reaction_types"))
)
 
# most_common_reaction: rank reactions by frequency within each drug, keep rank 1
reaction_freq = (
    df_drug_reactions
    .groupBy("drug_name", "reaction")
    .agg(F.count("*").alias("reaction_count"))
)
 
reaction_rank_window = Window.partitionBy("drug_name").orderBy(
    F.col("reaction_count").desc(), F.col("reaction").asc()  # tie-break alphabetically for determinism
)
 
df_most_common_reaction = (
    reaction_freq
    .withColumn("reaction_rank", F.row_number().over(reaction_rank_window))
    .filter(F.col("reaction_rank") == 1)
    .select("drug_name", F.col("reaction").alias("most_common_reaction"))
)
 
# --- outcome-level metrics: fatal_outcome_count + hospitalization_count ----------------
# Join drug reports to outcomes separately from the reaction join above, so the reaction
# fan-out doesn't inflate outcome counts. Dedup report+drug+outcome combos first.
df_drug_outcomes = (
    df_drug_reports
    .join(df_outcome, on=["primary_id", "case_id"], how="inner")
    .dropDuplicates(["primary_id", "drug_name", "outcome"])
)
 
df_outcome_counts = (
    df_drug_outcomes
    .groupBy("drug_name")
    .agg(
        F.sum(F.when(F.col("outcome") == "DEATH", 1).otherwise(0)).alias("fatal_outcome_count"),
        F.sum(F.when(F.col("outcome") == "HOSPITALIZATION", 1).otherwise(0)).alias("hospitalization_count"),
    )
)
 
#------------------------------------------------------------------------------------------------------#

# ASSEMBLE THE SCORECARD #

 
df_scorecard = (
    df_total_events
    .join(df_unique_reactions, on="drug_name", how="left")
    .join(df_most_common_reaction, on="drug_name", how="left")
    .join(df_outcome_counts, on="drug_name", how="left")
    .fillna(0, subset=["unique_reaction_types", "fatal_outcome_count", "hospitalization_count"])
)
 
# safety_score_rank — rank 1 = highest risk. No weighting/scoring formula: drugs are
# sorted in tiers — fatal outcomes first (highest priority signal), hospitalization
# count as the tie-breaker, then total adverse event volume as the final tie-breaker.
rank_window = Window.orderBy(
    F.col("fatal_outcome_count").desc(),
    F.col("hospitalization_count").desc(),
    F.col("total_adverse_events").desc(),
)
 
df_scorecard = df_scorecard.withColumn(
    "safety_score_rank", F.dense_rank().over(rank_window)
)
 
# Final column order to match the scorecard spec
df_scorecard = df_scorecard.select(
    "drug_name",
    "total_adverse_events",
    "unique_reaction_types",
    "fatal_outcome_count",
    "hospitalization_count",
    "most_common_reaction",
    "safety_score_rank",
)
 
#------------------------------------------------------------------------------------------------------#

# WRITE TO GOLD #

 
df_scorecard.write.mode("overwrite").parquet(GOLD_OUTPUT_PATH)
 
print(f"drug_safety_scorecard gold table saved successfully — {df_scorecard.count()} drugs scored")

#------------------------------------------------------------------------------------------------------#

# SCENARIO_02: SCORECARD #
# Business Question: "Which patient demographics — age, gender, weight — are most vulnerable
#                      to adverse events for a given drug?"
# Used by: Clinical research teams, Doctors, Insurance companies

SILVER_DEMO_PATH = "/Volumes/workspace/silver/faers_parquet/demo"
SILVER_DRUG_PATH = "/Volumes/workspace/silver/faers_parquet/drug"
SILVER_REACTION_PATH = "/Volumes/workspace/silver/faers_parquet/reaction"
SILVER_OUTCOME_PATH = "/Volumes/workspace/silver/faers_parquet/outcome"
 
GOLD_OUTPUT_PATH = "/Volumes/workspace/gold/analytics/patient_risk_profile"
 
#------------------------------------------------------------------------------------------------------#

# READ FROM SILVER #

 
df_demo = spark.read.parquet(SILVER_DEMO_PATH).select(
    "primary_id", "case_id", "patient_age", "sex", "weight"
)
 
df_drug = spark.read.parquet(SILVER_DRUG_PATH).select(
    "primary_id", "case_id", "drug_name"
)
 
df_reaction = spark.read.parquet(SILVER_REACTION_PATH).select(
    "primary_id", "case_id", "reaction"
)
 
df_outcome = spark.read.parquet(SILVER_OUTCOME_PATH).select(
    "primary_id", "case_id", "outcome"
)
 
#------------------------------------------------------------------------------------------------------#

# BUCKET THE DEMOGRAPHIC DIMENSIONS #
 
df_demo_bucketed = df_demo.withColumn(
    "age_group",
    F.when(F.col("patient_age").isNull(), "Unknown")
     .when(F.col("patient_age") <= 18, "0-18")
     .when(F.col("patient_age") <= 40, "19-40")
     .when(F.col("patient_age") <= 60, "41-60")
     .otherwise("60+")
).withColumn(
    "gender",
    F.when(F.upper(F.trim(F.col("sex"))) == "M", "Male")
     .when(F.upper(F.trim(F.col("sex"))) == "F", "Female")
     .otherwise("Unknown")
).withColumn(
    "weight_range",
    F.when(F.col("weight").isNull(), "Unknown")
     .when(F.col("weight") < 60, "Under 60kg")
     .when(F.col("weight") <= 90, "60-90kg")
     .otherwise("90kg+")
).select("primary_id", "case_id", "age_group", "gender", "weight_range")
 
#------------------------------------------------------------------------------------------------------#

# GOLD TRANSFORMATION # 
 
# One row per (report, drug, demographic bucket) — the base population for this scorecard.
# Dedup first: a drug can appear more than once per report (different drug_sequence entries).
df_base = (
    df_demo_bucketed
    .join(df_drug, on=["primary_id", "case_id"], how="inner")
    .dropDuplicates(["primary_id", "drug_name", "age_group", "gender", "weight_range"])
)
 
# --- adverse_event_count: distinct reports per (drug, demographic group) --------------
df_event_counts = (
    df_base
    .groupBy("drug_name", "age_group", "gender", "weight_range")
    .agg(F.count("*").alias("adverse_event_count"))
)
 
# --- most_common_reaction: rank reactions by frequency within each (drug, demo group) --
# Join base to reactions separately (fan-out expected: one report can have several
# reactions). Dedup report+drug+demo+reaction combos so a repeated reaction entry
# within the same report isn't counted twice.
df_base_reactions = (
    df_base
    .join(df_reaction, on=["primary_id", "case_id"], how="inner")
    .dropDuplicates(["primary_id", "drug_name", "age_group", "gender", "weight_range", "reaction"])
)
 
reaction_freq = (
    df_base_reactions
    .groupBy("drug_name", "age_group", "gender", "weight_range", "reaction")
    .agg(F.count("*").alias("reaction_count"))
)
 
reaction_rank_window = Window.partitionBy(
    "drug_name", "age_group", "gender", "weight_range"
).orderBy(
    F.col("reaction_count").desc(), F.col("reaction").asc()  # tie-break alphabetically for determinism
)
 
df_most_common_reaction = (
    reaction_freq
    .withColumn("reaction_rank", F.row_number().over(reaction_rank_window))
    .filter(F.col("reaction_rank") == 1)
    .select(
        "drug_name", "age_group", "gender", "weight_range",
        F.col("reaction").alias("most_common_reaction")
    )
)
 
# --- outcome_severity: worst outcome observed per (drug, demo group) ------------------
# Join base to outcomes separately from the reaction join, so reaction fan-out can't
# distort severity. Dedup report+drug+demo+outcome combos first.
df_base_outcomes = (
    df_base
    .join(df_outcome, on=["primary_id", "case_id"], how="inner")
    .dropDuplicates(["primary_id", "drug_name", "age_group", "gender", "weight_range", "outcome"])
)
 
# Fatal > Hospitalised > Recovered — worst-case observed in the group wins.
# (Rank 1 = worst; adjust this mapping if your business wants a different priority order.)
SEVERITY_PRIORITY = {
    "DEATH": (1, "Fatal"),
    "LIFE THREATENING": (2, "Hospitalised"),
    "HOSPITALIZATION": (2, "Hospitalised"),
    "DISABILITY": (2, "Hospitalised"),
    "REQUIRED INTERVENTION": (2, "Hospitalised"),
    "CONGENITAL ANOMALY": (2, "Hospitalised"),
    "OTHER SERIOUS": (2, "Hospitalised"),
}
DEFAULT_SEVERITY_RANK = 3
DEFAULT_SEVERITY_LABEL = "Recovered"
 
severity_rank_expr = F.lit(DEFAULT_SEVERITY_RANK)
severity_label_expr = F.lit(DEFAULT_SEVERITY_LABEL)
for outcome_value, (rank, label) in SEVERITY_PRIORITY.items():
    severity_rank_expr = F.when(F.col("outcome") == outcome_value, rank).otherwise(severity_rank_expr)
    severity_label_expr = F.when(F.col("outcome") == outcome_value, label).otherwise(severity_label_expr)
 
df_base_outcomes = df_base_outcomes.withColumn(
    "severity_rank", severity_rank_expr
).withColumn(
    "severity_label", severity_label_expr
)
 
severity_window = Window.partitionBy(
    "drug_name", "age_group", "gender", "weight_range"
).orderBy(F.col("severity_rank").asc())  # rank 1 (worst) sorts first
 
df_outcome_severity = (
    df_base_outcomes
    .withColumn("severity_row_num", F.row_number().over(severity_window))
    .filter(F.col("severity_row_num") == 1)
    .select(
        "drug_name", "age_group", "gender", "weight_range",
        F.col("severity_label").alias("outcome_severity")
    )
)
 
# --- risk_index: this drug's events in the group / that demographic's total events ----
# across ALL drugs (standard proportional-reporting-style signal-detection ratio).
df_demo_population = (
    df_base
    .groupBy("age_group", "gender", "weight_range")
    .agg(F.countDistinct("primary_id").alias("demographic_population_all_drugs"))
)
 
#------------------------------------------------------------------------------------------------------#

# ASSEMBLE THE RISK PROFILE #
 
df_risk_profile = (
    df_event_counts
    .join(df_most_common_reaction, on=["drug_name", "age_group", "gender", "weight_range"], how="left")
    .join(df_outcome_severity, on=["drug_name", "age_group", "gender", "weight_range"], how="left")
    .join(df_demo_population, on=["age_group", "gender", "weight_range"], how="left")
    .withColumn(
        "risk_index",
        F.round(
            F.col("adverse_event_count") / F.col("demographic_population_all_drugs"),
            4
        )
    )
    .drop("demographic_population_all_drugs")
)
 
# Final column order to match the scorecard spec
df_risk_profile = df_risk_profile.select(
    "drug_name",
    "age_group",
    "gender",
    "weight_range",
    "adverse_event_count",
    "most_common_reaction",
    "outcome_severity",
    "risk_index",
)
 
#------------------------------------------------------------------------------------------------------#

# WRITE TO GOLD #
 
df_risk_profile.write.mode("overwrite").parquet(GOLD_OUTPUT_PATH)
 
print(f"patient_risk_profile gold table saved successfully — {df_risk_profile.count()} demographic-drug rows")
